# 戦略8: 低ボラティリティ戦略の検証

**作成日**: 2026-02-21 09:30

**目的**: 低ボラティリティ効果の検証と加重方式の比較

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("ライブラリインポート完了")

## 1. データ読み込み

In [ ]:
PROJECT_ROOT = Path(r'C:\Users\yongr\claude project\workspace')

# 価格データ読み込み
print("価格データ読み込み中...")
df_price = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/prices/daily_quotes_all.parquet')
df_price['date'] = pd.to_datetime(df_price['date'])

# 期間フィルタ（2017年以降）
df_price = df_price[df_price['date'] >= '2017-01-01'].copy()
print(f"価格データ: {len(df_price):,} 行")
print(f"期間: {df_price['date'].min().date()} ~ {df_price['date'].max().date()}")
print(f"銘柄数: {df_price['code'].nunique():,}")

# 財務データ読み込み（配当性向用）
print("\n財務データ読み込み中...")
df_fin = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/financials/statements_all.parquet')
df_fin['disclosed_date'] = pd.to_datetime(df_fin['disclosed_date'])
df_fin = df_fin[df_fin['disclosed_date'] >= '2017-01-01'].copy()

# 年次決算のみ
if 'fiscal_quarter' in df_fin.columns:
    df_fin = df_fin[df_fin['fiscal_quarter'] == 'FY'].copy()
    print(f"年次決算のみに限定")

# 必須カラムのみ
if 'PayoutRatioAnn' in df_fin.columns:
    df_fin = df_fin[['disclosed_date', 'code', 'PayoutRatioAnn']].copy()
    df_fin = df_fin.dropna(subset=['PayoutRatioAnn'])
    print(f"財務データ（配当性向あり）: {len(df_fin):,} 行")
else:
    print("警告: PayoutRatioAnn列が存在しません。配当性向戦略は実行できません。")
    df_fin = pd.DataFrame()

print("\nデータ読み込み完了")

## 2. 価格データのピボット化

In [ ]:
print("価格データをピボット化中...")
df_price_pivot = df_price.pivot(index='date', columns='code', values='adjusted_close')
print(f"ピボットテーブル: {df_price_pivot.shape[0]} 日 × {df_price_pivot.shape[1]} 銘柄")
print("ピボット化完了")

## 3. リバランス日生成（月次）

In [ ]:
# 月末営業日
trading_days = pd.DataFrame({'date': df_price_pivot.index})
trading_days['year'] = trading_days['date'].dt.year
trading_days['month'] = trading_days['date'].dt.month
rebalance_dates = trading_days.groupby(['year', 'month'])['date'].max().values
rebalance_dates = pd.Series(rebalance_dates).sort_values().reset_index(drop=True)

print(f"リバランス日数: {len(rebalance_dates)}")
print(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}")

## 4. ボラティリティ計算関数

In [ ]:
def calculate_volatility(prices_pivot, date, lookback=60):
    """
    各銘柄の過去60営業日のボラティリティを計算（年率換算）
    
    Parameters:
    -----------
    prices_pivot : DataFrame
        ピボット化された価格データ（index=date, columns=code, values=adjusted_close）
    date : Timestamp
        計算基準日
    lookback : int
        過去何営業日のデータを使用するか（デフォルト60日）
    
    Returns:
    --------
    Series : 各銘柄の年率ボラティリティ（欠損値あり）
    """
    # dateが価格データに存在しない場合
    if date not in prices_pivot.index:
        return pd.Series(dtype=float)
    
    # dateから過去lookback営業日の価格を取得
    end_idx = prices_pivot.index.get_loc(date)
    start_idx = max(0, end_idx - lookback)
    
    # 過去データ取得
    historical_prices = prices_pivot.iloc[start_idx:end_idx+1]
    
    # 最低データ数のチェック
    if len(historical_prices) < lookback:
        return pd.Series(dtype=float)
    
    # 日次リターン計算
    returns = historical_prices.pct_change().dropna()
    
    # 標準偏差（年率換算: √252）
    volatility = returns.std() * np.sqrt(252)
    
    # 異常値除外（ボラティリティ > 100%は異常と判断）
    volatility = volatility[volatility <= 1.0]
    
    return volatility

# テスト
test_date = rebalance_dates.iloc[10]
test_vol = calculate_volatility(df_price_pivot, test_date, lookback=60)
print(f"テスト: {test_date.date()}")
print(f"ボラティリティ計算銘柄数: {len(test_vol)}")
print(f"平均ボラティリティ: {test_vol.mean()*100:.2f}%")
print(f"最小ボラティリティ: {test_vol.min()*100:.2f}%")
print(f"最大ボラティリティ: {test_vol.max()*100:.2f}%")

## 5. 財務データの事前処理（配当性向）

In [ ]:
if len(df_fin) > 0:
    print("財務データの事前処理中（配当性向）...")
    
    fin_by_date = {}
    
    for i, rdate in enumerate(rebalance_dates):
        if i % 20 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)}")
        
        # その日までに開示された財務データ
        available = df_fin[df_fin['disclosed_date'] <= rdate].copy()
        
        # 各銘柄の最新データ
        latest = available.sort_values('disclosed_date').groupby('code').tail(1)
        latest = latest.set_index('code')[['PayoutRatioAnn']]
        
        fin_by_date[rdate] = latest
    
    print("財務データ事前処理完了")
else:
    print("財務データなし - 配当性向戦略はスキップします")
    fin_by_date = {}

## 6. 銘柄スクリーニング関数

In [ ]:
def screen_low_volatility(prices_pivot, volatility, n_stocks=20, weighting='equal'):
    """
    低ボラティリティ銘柄のスクリーニングと加重計算
    
    Parameters:
    -----------
    prices_pivot : DataFrame
        ピボット化された価格データ
    volatility : Series
        各銘柄のボラティリティ
    n_stocks : int
        選定銘柄数
    weighting : str
        加重方式（'equal': 等金額, 'inverse_vol': ボラティリティ逆数）
    
    Returns:
    --------
    DataFrame : 選定銘柄とその加重
    """
    if len(volatility) < n_stocks:
        return pd.DataFrame()
    
    # ボラティリティ下位N銘柄を選定
    selected_codes = volatility.nsmallest(n_stocks)
    
    # 加重計算
    if weighting == 'equal':
        weights = pd.Series(1.0 / n_stocks, index=selected_codes.index)
    elif weighting == 'inverse_vol':
        # ボラティリティ逆数
        inv_vol = 1.0 / selected_codes
        weights = inv_vol / inv_vol.sum()
    else:
        raise ValueError(f"不明な加重方式: {weighting}")
    
    result = pd.DataFrame({
        'code': selected_codes.index,
        'volatility': selected_codes.values,
        'weight': weights.values
    })
    
    return result

def screen_baseline(prices_pivot, n_stocks=20):
    """
    ベースライン: 全銘柄の等金額加重（データ制約により時価総額は使用不可）
    
    Parameters:
    -----------
    prices_pivot : DataFrame
        ピボット化された価格データ
    n_stocks : int
        選定銘柄数
    
    Returns:
    --------
    DataFrame : 選定銘柄とその加重
    """
    # ランダムにN銘柄選定（毎回異なる銘柄を選ばないよう、銘柄コード順で選定）
    all_codes = prices_pivot.columns.tolist()
    if len(all_codes) < n_stocks:
        return pd.DataFrame()
    
    # 先頭N銘柄を選定（簡易的なベースライン）
    selected_codes = all_codes[:n_stocks]
    
    result = pd.DataFrame({
        'code': selected_codes,
        'volatility': np.nan,
        'weight': 1.0 / n_stocks
    })
    
    return result

def screen_low_vol_high_payout(prices_pivot, volatility, payout_data, n_stocks=20):
    """
    低ボラティリティ × 高配当性向戦略
    
    Parameters:
    -----------
    prices_pivot : DataFrame
        ピボット化された価格データ
    volatility : Series
        各銘柄のボラティリティ
    payout_data : DataFrame
        配当性向データ（index=code, columns=['PayoutRatioAnn']）
    n_stocks : int
        選定銘柄数
    
    Returns:
    --------
    DataFrame : 選定銘柄とその加重
    """
    if len(volatility) == 0 or len(payout_data) == 0:
        return pd.DataFrame()
    
    # マージ
    merged = pd.DataFrame({
        'volatility': volatility,
        'payout': payout_data['PayoutRatioAnn']
    }).dropna()
    
    if len(merged) < n_stocks:
        return pd.DataFrame()
    
    # ボラティリティ下位50%
    vol_threshold = merged['volatility'].quantile(0.5)
    low_vol = merged[merged['volatility'] <= vol_threshold]
    
    if len(low_vol) == 0:
        return pd.DataFrame()
    
    # 配当性向上位50%
    payout_threshold = merged['payout'].quantile(0.5)
    high_payout = merged[merged['payout'] >= payout_threshold]
    
    if len(high_payout) == 0:
        return pd.DataFrame()
    
    # 交差（共通銘柄）
    candidates = low_vol.index.intersection(high_payout.index)
    
    if len(candidates) < n_stocks:
        # 候補が不足している場合、低ボラのみで選定
        candidates = low_vol.nsmallest(n_stocks, 'volatility').index
    else:
        # ボラティリティが低い順にN銘柄選定
        candidates = merged.loc[candidates].nsmallest(n_stocks, 'volatility').index
    
    selected = merged.loc[candidates]
    
    # 等金額加重
    weights = pd.Series(1.0 / len(selected), index=selected.index)
    
    result = pd.DataFrame({
        'code': selected.index,
        'volatility': selected['volatility'].values,
        'weight': weights.values
    })
    
    return result

# テスト
test_selected_equal = screen_low_volatility(df_price_pivot, test_vol, n_stocks=20, weighting='equal')
test_selected_inv = screen_low_volatility(df_price_pivot, test_vol, n_stocks=20, weighting='inverse_vol')

print(f"\n等金額加重: {len(test_selected_equal)} 銘柄選定")
print(test_selected_equal.head())
print(f"\nボラティリティ逆数加重: {len(test_selected_inv)} 銘柄選定")
print(test_selected_inv.head())
print(f"\n加重の合計（等金額）: {test_selected_equal['weight'].sum():.4f}")
print(f"加重の合計（逆数）: {test_selected_inv['weight'].sum():.4f}")

## 7. バックテスト関数

In [ ]:
def run_backtest(rebalance_dates, prices_pivot, screening_func, initial_cash=10_000_000, 
                 tax_rate=0.20315, unit=100, **screening_kwargs):
    """
    バックテスト実行関数
    
    Parameters:
    -----------
    rebalance_dates : Series
        リバランス日のリスト
    prices_pivot : DataFrame
        ピボット化された価格データ
    screening_func : callable
        銘柄スクリーニング関数
    initial_cash : float
        初期資本
    tax_rate : float
        譲渡益税率
    unit : int
        単元株数
    screening_kwargs : dict
        スクリーニング関数に渡す追加引数
    
    Returns:
    --------
    DataFrame : バックテスト結果
    """
    cash = initial_cash
    portfolio = {}  # {code: {'shares': int, 'buy_price': float}}
    annual_realized_pnl = 0
    current_year = None
    results = []
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if i % 20 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * tax_rate
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 既存ポートフォリオ売却
        sell_value = 0
        if rebalance_date in prices_pivot.index:
            for code, position in portfolio.items():
                if code in prices_pivot.columns:
                    sell_price = prices_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # 銘柄選定
        selected = screening_func(prices_pivot, **screening_kwargs)
        
        if len(selected) == 0:
            results.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0,
                'annual_pnl': annual_realized_pnl,
                'portfolio': {}
            })
            continue
        
        # 購入（加重に基づいて）
        total_invested = 0
        
        for _, row in selected.iterrows():
            code = row['code']
            weight = row['weight']
            
            if code not in prices_pivot.columns:
                continue
            
            price = prices_pivot.loc[rebalance_date, code]
            if pd.isna(price):
                continue
            
            # 加重に基づいた投資金額
            target_amount = cash * weight
            shares = int(target_amount / (price * unit)) * unit
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested,
            'annual_pnl': annual_realized_pnl,
            'portfolio': portfolio.copy()
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * tax_rate
        cash -= tax
        results[-1]['cash'] = cash
    
    # 最終日の時価評価
    final_date = prices_pivot.index.max()
    final_portfolio_value = 0
    
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in prices_pivot.columns:
                final_price = prices_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        results[-1]['invested'] = final_portfolio_value
    
    return pd.DataFrame(results)

print("バックテスト関数定義完了")

## 8. バックテスト実行

### 8.1 ベースライン（市場全体）

In [ ]:
print("="*60)
print("バックテスト1: ベースライン（市場全体）")
print("="*60)

# ベースライン用のラッパー関数
def baseline_screening(prices_pivot, rebalance_date):
    return screen_baseline(prices_pivot, n_stocks=20)

# ボラティリティとリバランス日を含めたラッパー
def run_baseline_backtest():
    results_list = []
    cash = 10_000_000
    portfolio = {}
    annual_realized_pnl = 0
    current_year = None
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if i % 20 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * 0.20315
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 売却
        sell_value = 0
        if rebalance_date in df_price_pivot.index:
            for code, position in portfolio.items():
                if code in df_price_pivot.columns:
                    sell_price = df_price_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # 銘柄選定
        selected = screen_baseline(df_price_pivot, n_stocks=20)
        
        if len(selected) == 0:
            results_list.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0
            })
            continue
        
        # 購入
        total_invested = 0
        for _, row in selected.iterrows():
            code = row['code']
            weight = row['weight']
            
            if code not in df_price_pivot.columns:
                continue
            
            price = df_price_pivot.loc[rebalance_date, code]
            if pd.isna(price):
                continue
            
            target_amount = cash * weight
            shares = int(target_amount / (price * 100)) * 100
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results_list.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * 0.20315
        cash -= tax
        results_list[-1]['cash'] = cash
    
    # 最終日の時価評価
    final_date = df_price_pivot.index.max()
    final_portfolio_value = 0
    
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in df_price_pivot.columns:
                final_price = df_price_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        results_list[-1]['invested'] = final_portfolio_value
    
    return pd.DataFrame(results_list)

results_baseline = run_baseline_backtest()
print("\nベースライン完了")

### 8.2 低ボラティリティ（等金額加重）

In [ ]:
print("="*60)
print("バックテスト2: 低ボラティリティ（等金額加重）")
print("="*60)

def run_low_vol_equal_backtest():
    results_list = []
    cash = 10_000_000
    portfolio = {}
    annual_realized_pnl = 0
    current_year = None
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if i % 20 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * 0.20315
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 売却
        sell_value = 0
        if rebalance_date in df_price_pivot.index:
            for code, position in portfolio.items():
                if code in df_price_pivot.columns:
                    sell_price = df_price_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # ボラティリティ計算
        volatility = calculate_volatility(df_price_pivot, rebalance_date, lookback=60)
        
        # 銘柄選定
        selected = screen_low_volatility(df_price_pivot, volatility, n_stocks=20, weighting='equal')
        
        if len(selected) == 0:
            results_list.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0
            })
            continue
        
        # 購入
        total_invested = 0
        for _, row in selected.iterrows():
            code = row['code']
            weight = row['weight']
            
            if code not in df_price_pivot.columns:
                continue
            
            price = df_price_pivot.loc[rebalance_date, code]
            if pd.isna(price):
                continue
            
            target_amount = cash * weight
            shares = int(target_amount / (price * 100)) * 100
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results_list.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * 0.20315
        cash -= tax
        results_list[-1]['cash'] = cash
    
    # 最終日の時価評価
    final_date = df_price_pivot.index.max()
    final_portfolio_value = 0
    
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in df_price_pivot.columns:
                final_price = df_price_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        results_list[-1]['invested'] = final_portfolio_value
    
    return pd.DataFrame(results_list)

results_low_vol_equal = run_low_vol_equal_backtest()
print("\n低ボラティリティ（等金額）完了")

### 8.3 低ボラティリティ（ボラティリティ逆数加重）

In [ ]:
print("="*60)
print("バックテスト3: 低ボラティリティ（ボラティリティ逆数加重）")
print("="*60)

def run_low_vol_inv_backtest():
    results_list = []
    cash = 10_000_000
    portfolio = {}
    annual_realized_pnl = 0
    current_year = None
    
    for i, rebalance_date in enumerate(rebalance_dates):
        if i % 20 == 0:
            print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
        
        # 年の切り替わり
        if current_year != rebalance_date.year:
            if current_year is not None and annual_realized_pnl > 0:
                tax = annual_realized_pnl * 0.20315
                cash -= tax
            annual_realized_pnl = 0
            current_year = rebalance_date.year
        
        # 売却
        sell_value = 0
        if rebalance_date in df_price_pivot.index:
            for code, position in portfolio.items():
                if code in df_price_pivot.columns:
                    sell_price = df_price_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        annual_realized_pnl += pnl
        
        cash += sell_value
        portfolio = {}
        
        # ボラティリティ計算
        volatility = calculate_volatility(df_price_pivot, rebalance_date, lookback=60)
        
        # 銘柄選定
        selected = screen_low_volatility(df_price_pivot, volatility, n_stocks=20, weighting='inverse_vol')
        
        if len(selected) == 0:
            results_list.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': 0,
                'invested': 0
            })
            continue
        
        # 購入
        total_invested = 0
        for _, row in selected.iterrows():
            code = row['code']
            weight = row['weight']
            
            if code not in df_price_pivot.columns:
                continue
            
            price = df_price_pivot.loc[rebalance_date, code]
            if pd.isna(price):
                continue
            
            target_amount = cash * weight
            shares = int(target_amount / (price * 100)) * 100
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                portfolio[code] = {'shares': shares, 'buy_price': price}
        
        cash -= total_invested
        
        results_list.append({
            'date': rebalance_date,
            'cash': cash,
            'n_stocks': len(portfolio),
            'invested': total_invested
        })
    
    # 最終税金
    if annual_realized_pnl > 0:
        tax = annual_realized_pnl * 0.20315
        cash -= tax
        results_list[-1]['cash'] = cash
    
    # 最終日の時価評価
    final_date = df_price_pivot.index.max()
    final_portfolio_value = 0
    
    if len(portfolio) > 0:
        for code, position in portfolio.items():
            if code in df_price_pivot.columns:
                final_price = df_price_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        results_list[-1]['invested'] = final_portfolio_value
    
    return pd.DataFrame(results_list)

results_low_vol_inv = run_low_vol_inv_backtest()
print("\n低ボラティリティ（逆数加重）完了")

### 8.4 低ボラティリティ × 高配当性向

In [ ]:
if len(fin_by_date) > 0:
    print("="*60)
    print("バックテスト4: 低ボラティリティ × 高配当性向")
    print("="*60)
    
    def run_low_vol_payout_backtest():
        results_list = []
        cash = 10_000_000
        portfolio = {}
        annual_realized_pnl = 0
        current_year = None
        
        for i, rebalance_date in enumerate(rebalance_dates):
            if i % 20 == 0:
                print(f"  進捗: {i}/{len(rebalance_dates)} - {rebalance_date.date()}")
            
            # 年の切り替わり
            if current_year != rebalance_date.year:
                if current_year is not None and annual_realized_pnl > 0:
                    tax = annual_realized_pnl * 0.20315
                    cash -= tax
                annual_realized_pnl = 0
                current_year = rebalance_date.year
            
            # 売却
            sell_value = 0
            if rebalance_date in df_price_pivot.index:
                for code, position in portfolio.items():
                    if code in df_price_pivot.columns:
                        sell_price = df_price_pivot.loc[rebalance_date, code]
                        if pd.notna(sell_price):
                            sell_amount = position['shares'] * sell_price
                            sell_value += sell_amount
                            pnl = (sell_price - position['buy_price']) * position['shares']
                            annual_realized_pnl += pnl
            
            cash += sell_value
            portfolio = {}
            
            # ボラティリティ計算
            volatility = calculate_volatility(df_price_pivot, rebalance_date, lookback=60)
            
            # 配当性向データ取得
            payout_data = fin_by_date.get(rebalance_date, pd.DataFrame())
            
            # 銘柄選定
            selected = screen_low_vol_high_payout(df_price_pivot, volatility, payout_data, n_stocks=20)
            
            if len(selected) == 0:
                results_list.append({
                    'date': rebalance_date,
                    'cash': cash,
                    'n_stocks': 0,
                    'invested': 0
                })
                continue
            
            # 購入
            total_invested = 0
            for _, row in selected.iterrows():
                code = row['code']
                weight = row['weight']
                
                if code not in df_price_pivot.columns:
                    continue
                
                price = df_price_pivot.loc[rebalance_date, code]
                if pd.isna(price):
                    continue
                
                target_amount = cash * weight
                shares = int(target_amount / (price * 100)) * 100
                
                if shares > 0:
                    invest_amount = shares * price
                    total_invested += invest_amount
                    portfolio[code] = {'shares': shares, 'buy_price': price}
            
            cash -= total_invested
            
            results_list.append({
                'date': rebalance_date,
                'cash': cash,
                'n_stocks': len(portfolio),
                'invested': total_invested
            })
        
        # 最終税金
        if annual_realized_pnl > 0:
            tax = annual_realized_pnl * 0.20315
            cash -= tax
            results_list[-1]['cash'] = cash
        
        # 最終日の時価評価
        final_date = df_price_pivot.index.max()
        final_portfolio_value = 0
        
        if len(portfolio) > 0:
            for code, position in portfolio.items():
                if code in df_price_pivot.columns:
                    final_price = df_price_pivot.loc[final_date, code]
                    if pd.notna(final_price):
                        final_portfolio_value += position['shares'] * final_price
            results_list[-1]['invested'] = final_portfolio_value
        
        return pd.DataFrame(results_list)
    
    results_low_vol_payout = run_low_vol_payout_backtest()
    print("\n低ボラティリティ × 高配当性向 完了")
else:
    print("配当性向データなし - スキップ")
    results_low_vol_payout = None

## 9. パフォーマンス分析

In [ ]:
def analyze_performance(df_results, strategy_name, initial_cash=10_000_000):
    """
    パフォーマンス分析関数
    """
    df = df_results.copy()
    df['total_value'] = df['cash'] + df['invested']
    df['return'] = df['total_value'].pct_change()
    df['cumulative_return'] = (1 + df['return']).cumprod() - 1
    
    # 基本指標
    final_value = df['total_value'].iloc[-1]
    total_return = df['cumulative_return'].iloc[-1]
    
    # 年率換算
    years = (df['date'].iloc[-1] - df['date'].iloc[0]).days / 365.25
    annual_return = (1 + total_return) ** (1 / years) - 1
    
    # ボラティリティ（年率）
    annual_volatility = df['return'].std() * np.sqrt(12)
    
    # MDD
    df['peak'] = df['total_value'].cummax()
    df['drawdown'] = (df['total_value'] - df['peak']) / df['peak']
    mdd = df['drawdown'].min()
    
    # シャープレシオ
    if df['return'].std() > 0:
        sharpe = df['return'].mean() / df['return'].std() * np.sqrt(12)
    else:
        sharpe = 0
    
    # カルマー比
    if mdd < 0:
        calmar = annual_return / abs(mdd)
    else:
        calmar = 0
    
    result = {
        '戦略': strategy_name,
        '最終資産（円）': f"{final_value:,.0f}",
        '総リターン（%）': f"{total_return*100:.2f}",
        '年率リターン（%）': f"{annual_return*100:.2f}",
        '年率ボラティリティ（%）': f"{annual_volatility*100:.2f}",
        '最大DD（%）': f"{mdd*100:.2f}",
        'シャープレシオ': f"{sharpe:.2f}",
        'カルマー比': f"{calmar:.2f}"
    }
    
    return result, df

# 各戦略の分析
metrics_list = []

print("="*80)
print("パフォーマンス分析")
print("="*80)

result1, df_baseline_analyzed = analyze_performance(results_baseline, 'ベースライン（市場全体）')
metrics_list.append(result1)
print(f"\n{result1['戦略']}")
for k, v in result1.items():
    if k != '戦略':
        print(f"  {k}: {v}")

result2, df_low_vol_equal_analyzed = analyze_performance(results_low_vol_equal, '低ボラ（等金額）')
metrics_list.append(result2)
print(f"\n{result2['戦略']}")
for k, v in result2.items():
    if k != '戦略':
        print(f"  {k}: {v}")

result3, df_low_vol_inv_analyzed = analyze_performance(results_low_vol_inv, '低ボラ（逆数加重）')
metrics_list.append(result3)
print(f"\n{result3['戦略']}")
for k, v in result3.items():
    if k != '戦略':
        print(f"  {k}: {v}")

if results_low_vol_payout is not None:
    result4, df_low_vol_payout_analyzed = analyze_performance(results_low_vol_payout, '低ボラ × 高配当性向')
    metrics_list.append(result4)
    print(f"\n{result4['戦略']}")
    for k, v in result4.items():
        if k != '戦略':
            print(f"  {k}: {v}")

# 比較表
df_metrics = pd.DataFrame(metrics_list)
print("\n")
print("="*80)
print("戦略比較表")
print("="*80)
display(df_metrics)

print("\nパフォーマンス分析完了")

## 10. 加重方式の比較表

In [ ]:
# 等金額 vs 逆数加重の比較
weighting_comparison = pd.DataFrame([
    {
        '加重方式': '等金額加重',
        '最終資産（円）': result2['最終資産（円）'],
        '年率リターン（%）': result2['年率リターン（%）'],
        'シャープレシオ': result2['シャープレシオ'],
        'カルマー比': result2['カルマー比'],
        '最大DD（%）': result2['最大DD（%）']
    },
    {
        '加重方式': 'ボラティリティ逆数加重',
        '最終資産（円）': result3['最終資産（円）'],
        '年率リターン（%）': result3['年率リターン（%）'],
        'シャープレシオ': result3['シャープレシオ'],
        'カルマー比': result3['カルマー比'],
        '最大DD（%）': result3['最大DD（%）']
    }
])

print("="*80)
print("加重方式の比較")
print("="*80)
display(weighting_comparison)

# CSV保存
weighting_comparison.to_csv(
    PROJECT_ROOT / 'analyses/20260221_0930_quants_model_low_volatility/weighting_comparison.csv',
    index=False,
    encoding='utf-8-sig'
)
print("\n加重方式比較表を保存")

## 11. 結果の保存

In [ ]:
output_dir = PROJECT_ROOT / 'analyses/20260221_0930_quants_model_low_volatility'

# バックテスト結果（すべての戦略を1つのCSVに）
df_all_results = pd.DataFrame({
    'date': df_baseline_analyzed['date'],
    'baseline_value': df_baseline_analyzed['total_value'],
    'low_vol_equal_value': df_low_vol_equal_analyzed['total_value'],
    'low_vol_inv_value': df_low_vol_inv_analyzed['total_value']
})

if results_low_vol_payout is not None:
    df_all_results['low_vol_payout_value'] = df_low_vol_payout_analyzed['total_value']

df_all_results.to_csv(output_dir / 'backtest_results.csv', index=False, encoding='utf-8-sig')
print(f"バックテスト結果を保存: {output_dir / 'backtest_results.csv'}")

# 評価指標（JSON）
import json

metrics_dict = {item['戦略']: {k: v for k, v in item.items() if k != '戦略'} for item in metrics_list}

with open(output_dir / 'backtest_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_dict, f, ensure_ascii=False, indent=2)

print(f"評価指標を保存: {output_dir / 'backtest_metrics.json'}")

# サマリ（テキスト）
with open(output_dir / 'performance_summary.txt', 'w', encoding='utf-8') as f:
    f.write("低ボラティリティ戦略 バックテスト結果\n")
    f.write("="*80 + "\n\n")
    
    f.write("期間: 2017-01-01 ~ 2026-02-17\n")
    f.write("初期資本: 10,000,000円\n")
    f.write("リバランス頻度: 月次\n\n")
    
    for item in metrics_list:
        f.write(f"【{item['戦略']}】\n")
        for k, v in item.items():
            if k != '戦略':
                f.write(f"  {k}: {v}\n")
        f.write("\n")
    
    f.write("="*80 + "\n")
    f.write("結論:\n")
    f.write("- 詳細は docs/knowledges/20260221_0930_quants_model_low_volatility.md を参照\n")

print(f"サマリを保存: {output_dir / 'performance_summary.txt'}")

print("\n全ての結果を保存完了")